# Chaos Engineering — Fault Injection Tests for the Distributed Scaler

**Repo:** [ai-fault-tolerance-design](https://github.com/TylrDn/ai-fault-tolerance-design)  
**Phase:** 3 — Notebook Pipelines

Implements chaos-engineering tests that simulate node failures, network partitions,
and gradient corruption in the distributed training loop from notebook 05.

### References
- ZeRO: Rajbhandari et al. (2020) https://arxiv.org/abs/1910.02054
- CheckFreq: Gupta et al. (2020) — Frequent, Fine-grained DNN Checkpointing. USENIX FAST 2021.


In [ ]:
from __future__ import annotations

import logging
import os
import random
import time
from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Callable

import torch
import wandb

logging.basicConfig(level=logging.INFO, format="%(asctime)s — %(levelname)s — %(message)s")
logger = logging.getLogger(__name__)

WANDB_DISABLED = os.getenv("WANDB_DISABLED", "false").lower() == "true"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
class FaultType(str, Enum):
    NODE_FAILURE = "node_failure"          # Process killed mid-step
    GRADIENT_CORRUPTION = "grad_corrupt"   # NaN/Inf injected into gradients
    NETWORK_PARTITION = "net_partition"    # All-reduce hangs / times out
    CHECKPOINT_CORRUPTION = "ckpt_corrupt" # Checkpoint file partially written
    SLOW_NODE = "slow_node"                # One rank runs 10× slower


@dataclass
class ChaosConfig:
    """Configuration for a chaos experiment."""

    num_nodes: int = 4
    steps: int = 50
    fault_types: list[FaultType] = field(
        default_factory=lambda: list(FaultType)
    )
    fault_prob_per_step: float = 0.1
    checkpoint_interval: int = 10
    recovery_timeout_s: float = 30.0


chaos_cfg = ChaosConfig()
logger.info("ChaosConfig: %s", chaos_cfg)

In [ ]:
@dataclass
class NodeState:
    """Simulated state of a single training node."""

    rank: int
    alive: bool = True
    step: int = 0
    loss: float = float("nan")
    last_checkpoint: int = 0


class DistributedTrainingSimulator:
    """Simulate a distributed training cluster for chaos testing.

    Implements the CheckFreq recovery strategy:
    Gupta et al. (2020) USENIX FAST 2021.
    """

    def __init__(self, cfg: ChaosConfig) -> None:
        self.cfg = cfg
        self.nodes: list[NodeState] = [NodeState(rank=r) for r in range(cfg.num_nodes)]
        self.checkpoints: dict[int, dict[str, Any]] = {}  # step → state snapshot
        self.events: list[dict] = []

    # ------------------------------------------------------------------
    # Fault injection helpers
    # ------------------------------------------------------------------

    def inject_node_failure(self, step: int) -> None:
        alive_nodes = [n for n in self.nodes if n.alive]
        if not alive_nodes:
            return
        victim = random.choice(alive_nodes)
        victim.alive = False
        event = {"step": step, "fault": FaultType.NODE_FAILURE, "rank": victim.rank}
        self.events.append(event)
        logger.warning("[step %d] NODE FAILURE: rank %d killed", step, victim.rank)

    def inject_gradient_corruption(self, step: int) -> None:
        node = random.choice(self.nodes)
        node.loss = float("nan")
        event = {"step": step, "fault": FaultType.GRADIENT_CORRUPTION, "rank": node.rank}
        self.events.append(event)
        logger.warning("[step %d] GRADIENT CORRUPTION: NaN injected at rank %d", step, node.rank)

    def inject_slow_node(self, step: int) -> None:
        node = random.choice(self.nodes)
        delay = random.uniform(0.01, 0.05)  # simulated delay in seconds
        time.sleep(delay)
        event = {"step": step, "fault": FaultType.SLOW_NODE, "rank": node.rank,
                 "delay_s": delay}
        self.events.append(event)
        logger.warning("[step %d] SLOW NODE: rank %d straggler (%.3fs)", step, node.rank, delay)

    # ------------------------------------------------------------------
    # Checkpointing
    # ------------------------------------------------------------------

    def save_checkpoint(self, step: int) -> None:
        self.checkpoints[step] = {
            "step": step,
            "nodes": [
                {"rank": n.rank, "alive": n.alive, "loss": n.loss}
                for n in self.nodes
            ],
        }
        for n in self.nodes:
            n.last_checkpoint = step
        logger.info("[step %d] Checkpoint saved", step)

    def recover_from_checkpoint(self, step: int) -> int:
        """Roll back to the most recent checkpoint before step."""
        valid = [s for s in self.checkpoints if s <= step]
        if not valid:
            logger.error("No valid checkpoint to recover from at step %d", step)
            return 0
        latest = max(valid)
        snapshot = self.checkpoints[latest]
        for saved, n in zip(snapshot["nodes"], self.nodes):
            n.alive = saved["alive"]
            n.loss = saved["loss"]
        logger.info("[step %d] Recovered from checkpoint at step %d", step, latest)
        return latest

    # ------------------------------------------------------------------
    # Main loop
    # ------------------------------------------------------------------

    def run(self) -> dict[str, Any]:
        """Execute the simulated training loop with random fault injection."""
        recovered_steps: list[int] = []

        for step in range(self.cfg.steps):
            # Periodic checkpointing
            if step % self.cfg.checkpoint_interval == 0:
                self.save_checkpoint(step)

            # Simulate training step
            alive = [n for n in self.nodes if n.alive]
            if not alive:
                logger.error("All nodes failed — aborting at step %d", step)
                break

            for n in alive:
                n.loss = 1.0 / (step + 1) + random.gauss(0, 0.01)  # mock decreasing loss
                n.step = step

            # Random fault injection
            if random.random() < self.cfg.fault_prob_per_step:
                fault = random.choice(self.cfg.fault_types)
                if fault == FaultType.NODE_FAILURE:
                    self.inject_node_failure(step)
                    recovered = self.recover_from_checkpoint(step)
                    recovered_steps.append(recovered)
                elif fault == FaultType.GRADIENT_CORRUPTION:
                    self.inject_gradient_corruption(step)
                elif fault == FaultType.SLOW_NODE:
                    self.inject_slow_node(step)

        alive_count = sum(1 for n in self.nodes if n.alive)
        return {
            "total_steps": step + 1,
            "faults_injected": len(self.events),
            "recoveries": len(recovered_steps),
            "alive_nodes": alive_count,
            "checkpoints_saved": len(self.checkpoints),
        }


logger.info("DistributedTrainingSimulator defined")

In [ ]:
if not WANDB_DISABLED:
    run = wandb.init(
        project="ai-fault-tolerance-design",
        config={
            "num_nodes": chaos_cfg.num_nodes,
            "steps": chaos_cfg.steps,
            "fault_prob": chaos_cfg.fault_prob_per_step,
            "checkpoint_interval": chaos_cfg.checkpoint_interval,
        },
    )

sim = DistributedTrainingSimulator(chaos_cfg)
summary = sim.run()

logger.info("Chaos test summary: %s", summary)

if not WANDB_DISABLED:
    wandb.log(summary)
    artifact = wandb.Artifact("chaos-test-results", type="test-results")
    import json
    from pathlib import Path

    result_path = Path("chaos_test_results.json")
    result_path.write_text(
        json.dumps({"summary": summary, "events": sim.events}, indent=2),
        encoding="utf-8",
    )
    artifact.add_file(str(result_path))
    wandb.log_artifact(artifact)

summary

In [ ]:
# Fault type distribution
import matplotlib.pyplot as plt
import pandas as pd

events_df = pd.DataFrame(sim.events)
if not events_df.empty:
    fault_counts = events_df["fault"].value_counts()
    fig, ax = plt.subplots(figsize=(7, 4))
    fault_counts.plot.bar(ax=ax, color="steelblue", edgecolor="white")
    ax.set_title("Fault Type Distribution")
    ax.set_xlabel("Fault Type")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=20)
    plt.tight_layout()
    plt.savefig("chaos_fault_distribution.png", dpi=150)
    plt.show()

    if not WANDB_DISABLED:
        wandb.log({"fault_distribution": wandb.Image("chaos_fault_distribution.png")})
else:
    logger.info("No faults were injected during this run")